# 03 — Exploratory Data Analysis: Indian Liver Patient Dataset

This notebook performs a comprehensive EDA on the Indian Liver Patient Dataset (583 rows, 11 columns).

**Key preprocessing:**
- Map target `Dataset` column: `{1: 1, 2: 0}` (1 = liver patient, 0 = non-liver patient)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DATASETS, RAW_DATA_DIR, SEED
from src.data.loader import load_raw_dataset

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

## 1. Load Data

In [ ]:
df = load_raw_dataset('liver')
print(f"Shape: {df.shape}")
df.head()

## 2. Map Target Column

In [ ]:
# Map: 1 -> 1 (liver patient), 2 -> 0 (non-liver patient)
df['Dataset'] = df['Dataset'].map({1: 1, 2: 0})
print("Unique target values after mapping:", df['Dataset'].unique())
print(f"Value counts:\n{df['Dataset'].value_counts()}")

## 3. Basic Info and Summary Statistics

In [ ]:
df.info()

In [ ]:
df.describe()

## 4. Missing Values Heatmap

In [ ]:
print(f"Total missing values per column:\n{df.isnull().sum()}")

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=ax)
ax.set_title('Missing Values Heatmap — Liver Patient Dataset')
plt.tight_layout()
plt.show()

## 5. Class Balance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['Dataset'].value_counts().sort_index()
sns.barplot(x=counts.index, y=counts.values, ax=ax)
ax.set_xlabel('Dataset (0 = Non-Liver Patient, 1 = Liver Patient)')
ax.set_ylabel('Count')
ax.set_title('Class Distribution — Liver Patient')
for i, v in enumerate(counts.values):
    ax.text(i, v + 3, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap — Liver Patient Dataset')
plt.tight_layout()
plt.show()

## 7. Feature Distributions (KDE) Split by Target

In [ ]:
numeric_features = DATASETS['liver']['numeric_features']

n_cols = 3
n_rows = (len(numeric_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    ax = axes[i]
    for label in [0, 1]:
        subset = df[df['Dataset'] == label][col].dropna()
        sns.kdeplot(subset, ax=ax, label=f'Dataset={label}', fill=True, alpha=0.4)
    ax.set_title(f'{col} Distribution by Target')
    ax.legend()

for j in range(len(numeric_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('KDE Distributions of Numeric Features — Liver Patient', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Box Plots for Outlier Detection

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    ax = axes[i]
    sns.boxplot(x='Dataset', y=col, data=df, ax=ax)
    ax.set_title(f'{col} — Outliers')

for j in range(len(numeric_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Box Plots of Numeric Features — Liver Patient', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Pair Plot for Top Correlated Features

In [ ]:
top_features = corr['Dataset'].drop('Dataset').abs().sort_values(ascending=False).head(4).index.tolist()
print(f"Top correlated features: {top_features}")

pair_df = df[top_features + ['Dataset']].dropna()
g = sns.pairplot(pair_df, hue='Dataset', diag_kind='kde', corner=True,
                 plot_kws={'alpha': 0.5})
g.figure.suptitle('Pair Plot — Top Correlated Features with Target', y=1.02)
plt.show()

## Summary

Key findings from this EDA:
- The dataset is imbalanced: roughly 71% liver patients vs 29% non-liver patients.
- `Albumin_and_Globulin_Ratio` has a few missing values.
- Liver enzyme features (Alamine/Aspartate Aminotransferase) have heavy right-skewed distributions with outliers.
- Direct and Total Bilirubin are highly correlated with each other.
- Gender is the only categorical feature in the dataset.